# Phase 3 — Decomposed multi-head model

- Shared backbone: **128 → 64 → 32** (ReLU, batch norm, dropout)
- Heads: **P(play)** (`minutes > 0`), **P(60+)** (`minutes ≥ 60`), **P(goal | played)**, **P(assist | played)**, **P(clean sheet)**, **E[bonus]**, **E[goals conceded]** (DEF/GK)
- Appearance: `P(play) × (1 + P(60+))` → 0 if benched, 1 if sub, 2 if 60+ (FPL rules)
- **Recombine** head outputs into expected FPL points using official scoring rules by position

Run notebook `02` once first (`results/val_2024_25_predictions.csv`). 

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "fpl_model_dataset.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
BASELINE_METRICS_PATH = RESULTS_DIR / "phase3_model_comparison.csv"
BASELINE_VAL_PREDS_PATH = RESULTS_DIR / "val_2024_25_predictions.csv"

TRAIN_SEASONS = [
    "2016-17", "2017-18", "2018-19", "2019-20",
    "2020-21", "2021-22", "2022-23", "2023-24",
]
VAL_SEASON = "2024-25"
TARGET = "total_points"
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)

Project root: C:\FPL_project
Device: cpu


In [2]:
NUMERIC_FEATURES = [
    "total_points_roll3", "total_points_roll5",
    "minutes_roll3", "minutes_roll5",
    "goals_scored_roll3", "goals_scored_roll5",
    "assists_roll3", "assists_roll5",
    "expected_goals_roll3", "expected_goals_roll5",
    "expected_assists_roll3", "expected_assists_roll5",
    "team_goals_scored_gw_roll5", "team_goals_conceded_gw_roll5", "team_points_gw_roll5",
    "opponent_team_points_roll5", "opponent_team_gc_roll5",
    "last_season_ppg", "last_season_minutes_share",
    "was_home", "rest_days",
    "value", "selected", "transfers_in", "transfers_out", "transfers_balance",
]

POSITION_MAP = {"GK": 1, "GKP": 1, "DEF": 2, "MID": 3, "AM": 3, "FWD": 4}


def load_modeling_table(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["position_id"] = df["position"].map(POSITION_MAP).fillna(3).astype(int)
    df["is_promoted_team"] = df["is_promoted_team"].map({True: 1.0, False: 0.0}).fillna(0.0)
    df["was_home"] = pd.to_numeric(df["was_home"], errors="coerce").fillna(0.0)
    return df


def build_feature_matrix(df: pd.DataFrame) -> np.ndarray:
    x_num = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
    x_pos = df[["position_id"]].to_numpy(dtype=np.float32)
    x_prom = df[["is_promoted_team"]].to_numpy(dtype=np.float32)
    return np.hstack([x_num, x_pos, x_prom])


def split_sets(df: pd.DataFrame):
    train = df[df["season"].isin(TRAIN_SEASONS)].copy()
    val = df[df["season"] == VAL_SEASON].copy()
    return train, val


def scored_mask(df: pd.DataFrame) -> np.ndarray:
    return df["minutes"].fillna(0).to_numpy() > 0


def tensor_to_numpy(t: torch.Tensor) -> np.ndarray:
    return np.asarray(t.detach().cpu().tolist(), dtype=np.float32)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
        "spearman": spearmanr(y_true, y_pred).statistic,
    }


def eval_both_slices(y_true: np.ndarray, y_pred: np.ndarray, played_mask: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    all_m = regression_metrics(y_true, y_pred)
    played_m = regression_metrics(y_true[played_mask], y_pred[played_mask])
    return {
        "mae_all": all_m["mae"],
        "rmse_all": all_m["rmse"],
        "r2_all": all_m["r2"],
        "spearman_all": all_m["spearman"],
        "mae_played": played_m["mae"],
        "rmse_played": played_m["rmse"],
        "r2_played": played_m["r2"],
        "spearman_played": played_m["spearman"],
    }

MERGE_KEYS = ["season", "element", "gw", "player_id"]


def metrics_row(model: str, y_true: np.ndarray, y_pred: np.ndarray, played_mask: np.ndarray) -> dict:
    row = eval_both_slices(y_true, y_pred, played_mask)
    row["model"] = model
    return row


def build_comparison_table(
    val: pd.DataFrame,
    y_true: np.ndarray,
    played_mask: np.ndarray,
    pred_specs: dict[str, str | np.ndarray],
) -> pd.DataFrame:
    rows = []
    for model, spec in pred_specs.items():
        if isinstance(spec, str):
            pred = pd.to_numeric(val[spec], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
        else:
            pred = np.asarray(spec, dtype=np.float64)
        if len(pred) != len(y_true):
            raise ValueError(f"{model}: pred length {len(pred)} != val length {len(y_true)}")
        rows.append(metrics_row(model, y_true, pred, played_mask))
    return pd.DataFrame(rows)


def build_head_targets(df: pd.DataFrame) -> dict[str, np.ndarray]:
    # Sub-event labels for multi-task heads (same GW outcomes, not features).
    minutes = df["minutes"].fillna(0).to_numpy()
    starts = pd.to_numeric(df["starts"], errors="coerce")
    y_play = (minutes > 0).astype(np.float32)
    y_sixty = np.where(
        starts == 1,
        1.0,
        np.where(starts == 0, 0.0, (minutes >= 60).astype(float)),
    ).astype(np.float32)

    goals = pd.to_numeric(df["goals_scored"], errors="coerce").fillna(0).to_numpy()
    assists = pd.to_numeric(df["assists"], errors="coerce").fillna(0).to_numpy()
    cs = pd.to_numeric(df["clean_sheets"], errors="coerce").fillna(0).to_numpy()
    bonus = pd.to_numeric(df["bonus"], errors="coerce").fillna(0).to_numpy()
    gc = pd.to_numeric(df["goals_conceded"], errors="coerce").fillna(0).to_numpy()

    return {
        "play": y_play,
        "sixty": y_sixty,
        "goal": (goals > 0).astype(np.float32),
        "assist": (assists > 0).astype(np.float32),
        "cs": (cs > 0).astype(np.float32),
        "bonus": bonus.astype(np.float32),
        "gc": gc.astype(np.float32),
        "total_points": df[TARGET].to_numpy(dtype=np.float32),
        "position_id": df["position_id"].to_numpy(dtype=np.int64),
    }


df = load_modeling_table(DATA_PATH)
train_df, val_df = split_sets(df)
val_mask = scored_mask(val_df)
print(f"Train {len(train_df):,} | Val {len(val_df):,}")
print(f"Played rows — val {val_mask.mean():.1%}")

Train 196,538 | Val 27,605
Played rows — val 41.9%


## FPL scoring recombination

Maps head outputs to expected points using position-specific rules (appearance, goals, assists, CS, conceded, bonus).

In [ ]:
def expected_fpl_points(
    p_play: np.ndarray,
    p_sixty: np.ndarray,
    p_goal: np.ndarray,
    p_assist: np.ndarray,
    p_cs: np.ndarray,
    e_bonus: np.ndarray,
    e_gc: np.ndarray,
    position_id: np.ndarray,
) -> np.ndarray:
    """Combine decomposed head predictions into expected total_points."""
    pos = position_id.astype(int)
    goal_pts = np.select(
        [pos == 1, pos == 2, pos == 3, pos == 4],
        [6.0, 6.0, 5.0, 4.0],
        default=5.0,
    )
    cs_pts = np.select([pos <= 2, pos == 3], [4.0, 1.0], default=0.0)

    # FPL appearance: 0 if no play; 1 if 1-59 min; 2 if 60+.
    # E[pts | play] = P(60+) * 2 + (1 - P(60+)) * 1 = 1 + P(60+)
    appearance = p_play * (1.0 + p_sixty)

    goal_points = p_play * p_goal * goal_pts
    assist_points = p_play * p_assist * 3.0
    cs_points = p_play * p_sixty * p_cs * cs_pts
    gc_penalty = p_play * np.where(pos <= 2, -0.5 * np.clip(e_gc, 0.0, None), 0.0)
    bonus_points = p_play * e_bonus

    return (appearance + goal_points + assist_points + cs_points + gc_penalty + bonus_points).astype(np.float32)


GOAL_PTS_LUT = torch.tensor([5.0, 6.0, 6.0, 5.0, 4.0])
CS_PTS_LUT = torch.tensor([0.0, 4.0, 4.0, 1.0, 0.0])


def expected_fpl_points_torch(out: dict, position_id: torch.Tensor) -> torch.Tensor:
    pos = position_id.long().clamp(0, 4)
    goal_pts = GOAL_PTS_LUT.to(position_id.device)[pos]
    cs_pts = CS_PTS_LUT.to(position_id.device)[pos]
    p_play, p_sixty = out["play"], out["sixty"]
    appearance = p_play * (1.0 + p_sixty)
    goal_points = p_play * out["goal"] * goal_pts
    assist_points = p_play * out["assist"] * 3.0
    cs_points = p_play * p_sixty * out["cs"] * cs_pts
    gc_penalty = p_play * torch.where(pos <= 2, -0.5 * out["gc"], torch.zeros_like(out["gc"]))
    bonus_points = p_play * out["bonus"]
    return appearance + goal_points + assist_points + cs_points + gc_penalty + bonus_points


# check that expected_fpl_points matches the official scoring rules on some edge cases
pos = np.array([3])
print("Bench (no play):", expected_fpl_points(np.array([0.0]), np.array([0.0]), np.zeros(1), np.zeros(1), np.zeros(1), np.zeros(1), np.zeros(1), pos))
print("Sub blank:", expected_fpl_points(np.array([1.0]), np.array([0.0]), np.zeros(1), np.zeros(1), np.zeros(1), np.zeros(1), np.zeros(1), pos))
print("Sub scores:", expected_fpl_points(np.array([1.0]), np.array([0.0]), np.array([1.0]), np.zeros(1), np.zeros(1), np.zeros(1), np.zeros(1), pos))
print("Starter haul:", expected_fpl_points(np.array([1.0]), np.array([1.0]), np.array([0.4]), np.array([0.2]), np.array([0.0]), np.array([2.0]), np.array([0.0]), pos))

Bench (no play): [0.]
Sub blank: [1.]
Sub scores: [6.]
Starter haul: [6.6]


## Decomposed model + training

In [4]:
class DecomposedFPLNet(nn.Module):
    def __init__(self, in_dim: int, dropout: float = 0.25):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.head_play = nn.Linear(32, 1)
        self.head_sixty = nn.Linear(32, 1)
        self.head_goal = nn.Linear(32, 1)
        self.head_assist = nn.Linear(32, 1)
        self.head_cs = nn.Linear(32, 1)
        self.head_bonus = nn.Linear(32, 1)
        self.head_gc = nn.Linear(32, 1)

    def forward(self, x):
        h = self.backbone(x)
        return {
            "play": torch.sigmoid(self.head_play(h)).squeeze(-1),
            "sixty": torch.sigmoid(self.head_sixty(h)).squeeze(-1),
            "goal": torch.sigmoid(self.head_goal(h)).squeeze(-1),
            "assist": torch.sigmoid(self.head_assist(h)).squeeze(-1),
            "cs": torch.sigmoid(self.head_cs(h)).squeeze(-1),
            "bonus": F.relu(self.head_bonus(h)).squeeze(-1),
            "gc": F.relu(self.head_gc(h)).squeeze(-1),
        }


def heads_to_numpy(preds: dict) -> dict[str, np.ndarray]:
    return {k: tensor_to_numpy(v) for k, v in preds.items()}


def train_decomposed(
    X_train_s: np.ndarray,
    y_train: dict,
    X_val_s: np.ndarray,
    y_val: dict,
    val_played_mask: np.ndarray,
    epochs: int = 1000,
    batch_size: int = 4096,
    lr: float = 1e-3,
    patience: int = 6,
    aux_points_weight: float = 1.0,
    head_loss_weight: float = 0.5,
):
    in_dim = X_train_s.shape[1]
    model = DecomposedFPLNet(in_dim).to(DEVICE)

    pos_train = torch.tensor(y_train["position_id"], dtype=torch.long)
    pos_val = y_val["position_id"]

    train_ds = TensorDataset(
        torch.tensor(X_train_s, dtype=torch.float32),
        torch.tensor(y_train["play"], dtype=torch.float32),
        torch.tensor(y_train["sixty"], dtype=torch.float32),
        torch.tensor(y_train["goal"], dtype=torch.float32),
        torch.tensor(y_train["assist"], dtype=torch.float32),
        torch.tensor(y_train["cs"], dtype=torch.float32),
        torch.tensor(y_train["bonus"], dtype=torch.float32),
        torch.tensor(y_train["gc"], dtype=torch.float32),
        torch.tensor(y_train["total_points"], dtype=torch.float32),
        pos_train,
    )
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    X_val_t = torch.tensor(X_val_s, dtype=torch.float32, device=DEVICE)
    y_pts_val = y_val["total_points"]

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_state = None
    best_val_rho = -1.0
    stale = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for xb, y_play, y_sixty, y_goal, y_assist, y_cs, y_bonus, y_gc, y_pts, pos_b in loader:
            xb = xb.to(DEVICE)
            y_play, y_sixty = y_play.to(DEVICE), y_sixty.to(DEVICE)
            y_goal, y_assist = y_goal.to(DEVICE), y_assist.to(DEVICE)
            y_cs, y_bonus, y_gc, y_pts = y_cs.to(DEVICE), y_bonus.to(DEVICE), y_gc.to(DEVICE), y_pts.to(DEVICE)
            pos_b = pos_b.to(DEVICE)
            out = model(xb)

            loss_play = F.binary_cross_entropy(out["play"], y_play)
            loss_sixty = F.binary_cross_entropy(out["sixty"], y_sixty, reduction="none")
            loss_sixty = (loss_sixty * (0.15 + 0.85 * y_play)).mean()
            goal_loss = F.binary_cross_entropy(out["goal"], y_goal, reduction="none")
            assist_loss = F.binary_cross_entropy(out["assist"], y_assist, reduction="none")
            w_play = 0.15 + 0.85 * y_play
            loss_goal = (goal_loss * w_play).mean()
            loss_assist = (assist_loss * w_play).mean()

            cs_loss = F.binary_cross_entropy(out["cs"], y_cs, reduction="none")
            cs_mask = (pos_b <= 2) | (pos_b == 3)
            w_cs = y_play * (0.15 + 0.85 * y_sixty)
            loss_cs = (cs_loss * w_cs)[cs_mask].mean() if cs_mask.any() else (cs_loss * w_cs).mean()

            loss_bonus = F.mse_loss(out["bonus"], y_bonus)
            gc_loss = F.mse_loss(out["gc"], y_gc, reduction="none")
            def_gk = pos_b <= 2
            loss_gc = gc_loss[def_gk].mean() if def_gk.any() else torch.tensor(0.0, device=DEVICE)

            pred_pts = expected_fpl_points_torch(out, pos_b)
            loss_aux = F.mse_loss(pred_pts, y_pts)

            head_loss = loss_play + loss_sixty + loss_goal + loss_assist + loss_cs + loss_bonus + 0.25 * loss_gc
            loss = head_loss_weight * head_loss + aux_points_weight * loss_aux
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            out_val = model(X_val_t)
            pos_val_t = torch.as_tensor(pos_val.tolist(), dtype=torch.long, device=DEVICE)
            pred_val = tensor_to_numpy(expected_fpl_points_torch(out_val, pos_val_t))
            val_rho = spearmanr(y_pts_val, pred_val).statistic

        scheduler.step(val_rho)
        if val_rho > best_val_rho:
            best_val_rho = val_rho
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        out_val = model(X_val_t)
        pos_val_t = torch.as_tensor(pos_val.tolist(), dtype=torch.long, device=DEVICE)
        pred_val = tensor_to_numpy(expected_fpl_points_torch(out_val, pos_val_t))
    return model, pred_val, out_val


X_train = build_feature_matrix(train_df)
X_val = build_feature_matrix(val_df)
y_train = build_head_targets(train_df)
y_val = build_head_targets(val_df)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)

decomp_model, val_pred_decomposed, val_heads = train_decomposed(
    X_train_s, y_train, X_val_s, y_val, val_mask
)

decomp_metrics = metrics_row("mlp_decomposed", y_val["total_points"], val_pred_decomposed, val_mask)
print(
    "Trained | all: MAE={mae_all:.4f} rho={spearman_all:.4f} | "
    "played: MAE={mae_played:.4f} rho={spearman_played:.4f}".format(**decomp_metrics)
)

Trained | all: MAE=1.0404 rho=0.6923 | played: MAE=1.8664 rho=0.3684


## Compare to baselines (notebook 02)

In [5]:
y_true = y_val["total_points"]
pred_specs = {
    "roll5_points": "total_points_roll5",
    "last_season_ppg": "last_season_ppg",
    "mlp_decomposed": val_pred_decomposed,
}

if BASELINE_VAL_PREDS_PATH.exists():
    base_preds = pd.read_csv(BASELINE_VAL_PREDS_PATH)
    if len(base_preds) != len(val_df):
        raise ValueError("baseline preds length mismatch — re-run notebook 02")
    if not np.array_equal(base_preds[MERGE_KEYS].values, val_df[MERGE_KEYS].values):
        raise ValueError("baseline preds row order mismatch — re-run notebook 02")
    pred_specs["mlp_points"] = base_preds["pred_mlp"].to_numpy()
    pred_specs["ridge_linear"] = base_preds["pred_ridge"].to_numpy()
else:
    print("Run 02_phase3_baseline_model.ipynb first for MLP/ridge rows")

comparison = build_comparison_table(val_df, y_true, val_mask, pred_specs)

print("Played rows only (minutes > 0):")
display(comparison[["model", "mae_played", "spearman_played", "rmse_played", "r2_played"]].sort_values("spearman_played", ascending=False))

print("All validation rows (full player pool):")
comparison[["model", "mae_all", "spearman_all", "rmse_all", "r2_all"]].sort_values("spearman_all", ascending=False)

Played rows only (minutes > 0):


,model,mae_played,spearman_played,rmse_played,r2_played
2,mlp_decomposed,1.866383,0.368407,2.803010,0.062198
3,mlp_points,1.863091,0.366992,2.799487,0.064553
4,ridge_linear,1.870431,0.347047,2.840238,0.037121
0,roll5_points,2.081290,0.290270,3.067166,-0.122889
1,last_season_ppg,2.188108,0.226227,3.063853,-0.120464


All validation rows (full player pool):


,model,mae_all,spearman_all,rmse_all,r2_all
2,mlp_decomposed,1.040445,0.692297,2.059983,0.273506
3,mlp_points,1.056206,0.686446,2.059723,0.273690
0,roll5_points,1.090377,0.675553,2.177385,0.188338
4,ridge_linear,1.114626,0.644458,2.105387,0.241127
1,last_season_ppg,1.552684,0.326069,2.491475,-0.062718


In [6]:
val_preds = val_df[["season", "element", "gw", "player_id", "name", "minutes", TARGET]].copy()
val_preds["pred_decomposed"] = val_pred_decomposed

if BASELINE_VAL_PREDS_PATH.exists():
    base_preds = pd.read_csv(BASELINE_VAL_PREDS_PATH)
    val_preds["pred_mlp"] = base_preds["pred_mlp"].to_numpy()
    val_preds["pred_ridge"] = base_preds["pred_ridge"].to_numpy()
    y_true = val_preds[TARGET].to_numpy()
    print("Head-to-head on same val rows:")
    for label, col in [("MLP", "pred_mlp"), ("Decomposed", "pred_decomposed")]:
        m = eval_both_slices(y_true, val_preds[col].to_numpy(), val_mask)
        print(f"  {label} — all: MAE={m['mae_all']:.4f} rho={m['spearman_all']:.4f}")
        print(f"         played: MAE={m['mae_played']:.4f} rho={m['spearman_played']:.4f}")

Head-to-head on same val rows:
  MLP — all: MAE=1.0562 rho=0.6864
         played: MAE=1.8631 rho=0.3670
  Decomposed — all: MAE=1.0404 rho=0.6923
         played: MAE=1.8664 rho=0.3684


In [7]:
# Example head outputs 
with torch.no_grad():
    heads_np = heads_to_numpy(val_heads)

inspect = val_df[["name", "position", "gw", "minutes", TARGET]].copy()
prob_heads = ["play", "sixty", "goal", "assist", "cs"]
for k in prob_heads:
    inspect[f"p_{k}"] = heads_np[k]
inspect["e_bonus"] = heads_np["bonus"]
inspect["e_gc"] = heads_np["gc"]
inspect["pred_decomposed"] = val_pred_decomposed
inspect.loc[scored_mask(val_df)].nlargest(8, "pred_decomposed")

,name,position,gw,minutes,total_points,p_play,p_sixty,p_goal,p_assist,p_cs,e_bonus,e_gc,pred_decomposed
208982,Mohamed Salah,MID,20,90,7,0.999916,0.998155,0.460422,0.079415,0.289398,1.605961,2.825405,6.432798
208980,Mohamed Salah,MID,18,90,9,0.999849,0.997114,0.455939,0.095262,0.301346,1.516610,2.635633,6.378719
208978,Mohamed Salah,MID,16,90,5,0.999831,0.996986,0.459760,0.094890,0.291104,1.480340,2.635938,6.349952
208993,Mohamed Salah,MID,30,90,3,0.999679,0.995137,0.441086,0.125204,0.313591,1.400031,2.384269,6.286256
208976,Mohamed Salah,MID,13,83,13,0.999637,0.994921,0.453963,0.116197,0.306530,1.352300,2.408883,6.268318
208977,Mohamed Salah,MID,14,90,18,0.999687,0.995418,0.454920,0.110410,0.291941,1.371253,2.467349,6.261147
208981,Mohamed Salah,MID,19,90,16,0.999707,0.995566,0.455082,0.112939,0.280223,1.364763,2.479935,6.251703
208984,Mohamed Salah,MID,22,90,3,0.999602,0.994252,0.447355,0.125777,0.290739,1.353225,2.367608,6.248160


In [8]:
_save_specs = {
    "roll5_points": "total_points_roll5",
    "last_season_ppg": "last_season_ppg",
    "mlp_decomposed": val_pred_decomposed,
}
if BASELINE_VAL_PREDS_PATH.exists():
    _base = pd.read_csv(BASELINE_VAL_PREDS_PATH)
    _save_specs["mlp_points"] = _base["pred_mlp"].to_numpy()
    _save_specs["ridge_linear"] = _base["pred_ridge"].to_numpy()
comparison_save = build_comparison_table(val_df, y_val["total_points"], val_mask, _save_specs)
comparison_save.to_csv(RESULTS_DIR / "phase3_model_comparison_with_decomposed.csv", index=False)
print("Saved comparison:")
print(comparison_save[["model", "mae_all", "spearman_all", "mae_played", "spearman_played"]].to_string(index=False))

val_preds = val_df[["season", "element", "gw", "player_id", "name", "minutes", TARGET]].copy()
val_preds["pred_decomposed"] = val_pred_decomposed
if BASELINE_VAL_PREDS_PATH.exists():
    val_preds["pred_mlp"] = _base["pred_mlp"].to_numpy()
    val_preds["pred_ridge"] = _base["pred_ridge"].to_numpy()
assert len(val_preds) == len(val_df)
val_preds.to_csv(RESULTS_DIR / "val_2024_25_predictions_with_decomposed.csv", index=False)
torch.save(
    {
        "model_state": decomp_model.state_dict(),
        "in_dim": X_train_s.shape[1],
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
    },
    RESULTS_DIR / "phase3_decomposed_mlp.pt",
)
print("Saved comparison + predictions + weights to", RESULTS_DIR)

Saved comparison:
          model  mae_all  spearman_all  mae_played  spearman_played
   roll5_points 1.090377      0.675553    2.081290         0.290270
last_season_ppg 1.552684      0.326069    2.188108         0.226227
 mlp_decomposed 1.040445      0.692297    1.866383         0.368407
     mlp_points 1.056206      0.686446    1.863091         0.366992
   ridge_linear 1.114626      0.644458    1.870431         0.347047
Saved comparison + predictions + weights to C:\FPL_project\results


## Try different models + pick best settings

Step 1: try 4 layer sizes (quick, on 80k rows).  
Step 2: try 7 learning-rate/dropout combos on the best 2.  
Then heavy-train the winner in the next cell.


In [ ]:
import time


class DecomposedFPLNetConfig(nn.Module):
    """Decomposed model with configurable backbone width/depth."""

    def __init__(self, in_dim: int, hidden_dims: list[int], dropout: float = 0.25):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev = h
        self.backbone = nn.Sequential(*layers)
        self.head_play = nn.Linear(prev, 1)
        self.head_sixty = nn.Linear(prev, 1)
        self.head_goal = nn.Linear(prev, 1)
        self.head_assist = nn.Linear(prev, 1)
        self.head_cs = nn.Linear(prev, 1)
        self.head_bonus = nn.Linear(prev, 1)
        self.head_gc = nn.Linear(prev, 1)

    def forward(self, x):
        h = self.backbone(x)
        return {
            "play": torch.sigmoid(self.head_play(h)).squeeze(-1),
            "sixty": torch.sigmoid(self.head_sixty(h)).squeeze(-1),
            "goal": torch.sigmoid(self.head_goal(h)).squeeze(-1),
            "assist": torch.sigmoid(self.head_assist(h)).squeeze(-1),
            "cs": torch.sigmoid(self.head_cs(h)).squeeze(-1),
            "bonus": F.relu(self.head_bonus(h)).squeeze(-1),
            "gc": F.relu(self.head_gc(h)).squeeze(-1),
        }


def train_decomposed_config(
    model: nn.Module,
    X_train_s: np.ndarray,
    y_train: dict,
    X_val_s: np.ndarray,
    y_val: dict,
    val_played_mask: np.ndarray,
    epochs: int = 30,
    batch_size: int = 4096,
    lr: float = 1e-3,
    patience: int = 6,
    aux_points_weight: float = 1.0,
    head_loss_weight: float = 0.5,
    verbose: bool = False,
):
    model = model.to(DEVICE)
    pos_train = torch.tensor(y_train["position_id"], dtype=torch.long)
    pos_val = y_val["position_id"]

    train_ds = TensorDataset(
        torch.tensor(X_train_s, dtype=torch.float32),
        torch.tensor(y_train["play"], dtype=torch.float32),
        torch.tensor(y_train["sixty"], dtype=torch.float32),
        torch.tensor(y_train["goal"], dtype=torch.float32),
        torch.tensor(y_train["assist"], dtype=torch.float32),
        torch.tensor(y_train["cs"], dtype=torch.float32),
        torch.tensor(y_train["bonus"], dtype=torch.float32),
        torch.tensor(y_train["gc"], dtype=torch.float32),
        torch.tensor(y_train["total_points"], dtype=torch.float32),
        pos_train,
    )
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    X_val_t = torch.tensor(X_val_s, dtype=torch.float32, device=DEVICE)
    y_pts_val = y_val["total_points"]

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_state = None
    best_val_rho = -1.0
    stale = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        for xb, y_play, y_sixty, y_goal, y_assist, y_cs, y_bonus, y_gc, y_pts, pos_b in loader:
            xb = xb.to(DEVICE)
            y_play, y_sixty = y_play.to(DEVICE), y_sixty.to(DEVICE)
            y_goal, y_assist = y_goal.to(DEVICE), y_assist.to(DEVICE)
            y_cs, y_bonus, y_gc, y_pts = y_cs.to(DEVICE), y_bonus.to(DEVICE), y_gc.to(DEVICE), y_pts.to(DEVICE)
            pos_b = pos_b.to(DEVICE)
            out = model(xb)

            loss_play = F.binary_cross_entropy(out["play"], y_play)
            loss_sixty = F.binary_cross_entropy(out["sixty"], y_sixty, reduction="none")
            loss_sixty = (loss_sixty * (0.15 + 0.85 * y_play)).mean()
            goal_loss = F.binary_cross_entropy(out["goal"], y_goal, reduction="none")
            assist_loss = F.binary_cross_entropy(out["assist"], y_assist, reduction="none")
            w_play = 0.15 + 0.85 * y_play
            loss_goal = (goal_loss * w_play).mean()
            loss_assist = (assist_loss * w_play).mean()

            cs_loss = F.binary_cross_entropy(out["cs"], y_cs, reduction="none")
            cs_mask = (pos_b <= 2) | (pos_b == 3)
            w_cs = y_play * (0.15 + 0.85 * y_sixty)
            loss_cs = (cs_loss * w_cs)[cs_mask].mean() if cs_mask.any() else (cs_loss * w_cs).mean()

            loss_bonus = F.mse_loss(out["bonus"], y_bonus)
            gc_loss = F.mse_loss(out["gc"], y_gc, reduction="none")
            def_gk = pos_b <= 2
            loss_gc = gc_loss[def_gk].mean() if def_gk.any() else torch.tensor(0.0, device=DEVICE)

            pred_pts = expected_fpl_points_torch(out, pos_b)
            loss_aux = F.mse_loss(pred_pts, y_pts)

            head_loss = loss_play + loss_sixty + loss_goal + loss_assist + loss_cs + loss_bonus + 0.25 * loss_gc
            loss = head_loss_weight * head_loss + aux_points_weight * loss_aux
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            out_val = model(X_val_t)
            pos_val_t = torch.as_tensor(pos_val.tolist(), dtype=torch.long, device=DEVICE)
            pred_val = tensor_to_numpy(expected_fpl_points_torch(out_val, pos_val_t))
            val_rho = spearmanr(y_pts_val, pred_val).statistic
            val_mae = mean_absolute_error(y_pts_val, pred_val)

        scheduler.step(val_rho)
        history.append({"epoch": epoch, "val_spearman_all": val_rho, "val_mae_all": val_mae})

        if val_rho > best_val_rho:
            best_val_rho = val_rho
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        out_val = model(X_val_t)
        pos_val_t = torch.as_tensor(pos_val.tolist(), dtype=torch.long, device=DEVICE)
        pred_val = tensor_to_numpy(expected_fpl_points_torch(out_val, pos_val_t))

    metrics = eval_both_slices(y_pts_val, pred_val, val_played_mask)
    metrics["epochs_ran"] = len(history)
    if verbose:
        print(
            f"  rho_all={metrics['spearman_all']:.4f} MAE_all={metrics['mae_all']:.4f}"
            f"  epochs={metrics['epochs_ran']}"
        )

    return model, pred_val, metrics, history

In [ ]:
import time

# layer sizes to try
architectures = {
    "small": [64, 32],
    "default": [128, 64, 32],
    "wide": [256, 128, 64],
    "deep": [128, 128, 128, 64],
}

# settings we use for every architecture in step 1
lr = 1e-3
dropout = 0.25
batch_size = 4096
aux_weight = 1.0

in_dim = X_train_s.shape[1]
results = []

# step 1: pick best architecture 
np.random.seed(RANDOM_SEED)
sample_idx = np.random.choice(len(X_train_s), size=80_000, replace=False)
X_small = X_train_s[sample_idx]
y_small = {key: y_train[key][sample_idx] for key in y_train}

print("Step 1: try each architecture")
for name, layers in architectures.items():
    torch.manual_seed(RANDOM_SEED)
    model = DecomposedFPLNetConfig(in_dim, layers, dropout=dropout)
    model, _, metrics, _ = train_decomposed_config(
        model, X_small, y_small, X_val_s, y_val, val_mask,
        epochs=40, batch_size=batch_size, lr=lr,
        patience=3, aux_points_weight=aux_weight,
    )
    results.append({
        "step": 1,
        "name": name,
        "layers": str(layers),
        "lr": lr,
        "dropout": dropout,
        "aux_weight": aux_weight,
        **metrics,
    })
    print(f"  {name}: rho_all={metrics['spearman_all']:.4f} MAE_all={metrics['mae_all']:.4f}")

step1_df = pd.DataFrame(results).sort_values(["spearman_all", "mae_all"], ascending=[False, True])
best_two = step1_df.head(2)["name"].tolist()
print("Best two architectures:", best_two)

# step 2: tune hyperparams
hp_to_try = [
    {"lr": 5e-4, "dropout": 0.15, "aux_weight": 0.5},
    {"lr": 5e-4, "dropout": 0.25, "aux_weight": 1.0},
    {"lr": 1e-3, "dropout": 0.15, "aux_weight": 0.75},
    {"lr": 1e-3, "dropout": 0.25, "aux_weight": 1.0},
    {"lr": 1e-3, "dropout": 0.35, "aux_weight": 1.0},
    {"lr": 2e-3, "dropout": 0.25, "aux_weight": 1.0},
    {"lr": 2e-3, "dropout": 0.35, "aux_weight": 1.0},
]

print("\nStep 2: tune hyperparams on", best_two)
t0 = time.time()
for arch_name in best_two:
    layers = architectures[arch_name]
    for hp in hp_to_try:
        torch.manual_seed(RANDOM_SEED)
        model = DecomposedFPLNetConfig(in_dim, layers, dropout=hp["dropout"])
        model, _, metrics, _ = train_decomposed_config(
            model, X_train_s, y_train, X_val_s, y_val, val_mask,
            epochs=20, batch_size=batch_size, lr=hp["lr"],
            patience=5, aux_points_weight=hp["aux_weight"],
        )
        results.append({
            "step": 2,
            "name": arch_name,
            "layers": str(layers),
            **hp,
            **metrics,
        })
        print(f"  {arch_name} lr={hp['lr']} drop={hp['dropout']}: rho_all={metrics['spearman_all']:.4f}")

search_df = pd.DataFrame(results).sort_values(["spearman_all", "mae_all"], ascending=[False, True])
print(f"\nDone in {time.time() - t0:.0f}s, {len(results)} models tried")
search_df.head(10)

Step 1: try each architecture
  small: rho_all=0.6821 MAE_all=1.1546
  default: rho_all=0.6867 MAE_all=1.0660
  wide: rho_all=0.6870 MAE_all=1.0512
  deep: rho_all=0.6878 MAE_all=1.0491
Best two architectures: ['deep', 'wide']

Step 2: tune hyperparams on ['deep', 'wide']
  deep lr=0.0005 drop=0.15: rho_all=0.6928
  deep lr=0.0005 drop=0.25: rho_all=0.6904
  deep lr=0.001 drop=0.15: rho_all=0.6929
  deep lr=0.001 drop=0.25: rho_all=0.6929
  deep lr=0.001 drop=0.35: rho_all=0.6912
  deep lr=0.002 drop=0.25: rho_all=0.6925
  deep lr=0.002 drop=0.35: rho_all=0.6918
  wide lr=0.0005 drop=0.15: rho_all=0.6924
  wide lr=0.0005 drop=0.25: rho_all=0.6902
  wide lr=0.001 drop=0.15: rho_all=0.6920
  wide lr=0.001 drop=0.25: rho_all=0.6933
  wide lr=0.001 drop=0.35: rho_all=0.6922
  wide lr=0.002 drop=0.25: rho_all=0.6921
  wide lr=0.002 drop=0.35: rho_all=0.6932

Done in 2747s, 18 models tried


,step,name,layers,lr,dropout,aux_weight,mae_all,rmse_all,r2_all,spearman_all,mae_played,rmse_played,r2_played,spearman_played,epochs_ran
14,2,wide,"[256, 128, 64]",0.0010,0.25,1.00,1.058093,2.058182,0.274776,0.693303,1.886949,2.802772,0.062357,0.366724,20
17,2,wide,"[256, 128, 64]",0.0020,0.35,1.00,1.039352,2.058003,0.274902,0.693194,1.865301,2.802740,0.062378,0.369710,20
7,2,deep,"[128, 128, 128, 64]",0.0010,0.25,1.00,1.030084,2.054711,0.277220,0.692904,1.842712,2.812093,0.056110,0.369126,20
6,2,deep,"[128, 128, 128, 64]",0.0010,0.15,0.75,1.037250,2.057231,0.275445,0.692883,1.850917,2.811232,0.056687,0.366308,19
4,2,deep,"[128, 128, 128, 64]",0.0005,0.15,0.50,1.040238,2.053868,0.277812,0.692804,1.853289,2.809579,0.057797,0.367507,20
9,2,deep,"[128, 128, 128, 64]",0.0020,0.25,1.00,1.045814,2.056676,0.275837,0.692544,1.857627,2.801916,0.062929,0.367635,16
11,2,wide,"[256, 128, 64]",0.0005,0.15,0.50,1.069539,2.057276,0.275414,0.692416,1.886523,2.797707,0.065743,0.367294,20
15,2,wide,"[256, 128, 64]",0.0010,0.35,1.00,1.053423,2.055574,0.276612,0.692242,1.879839,2.802016,0.062863,0.365287,20
16,2,wide,"[256, 128, 64]",0.0020,0.25,1.00,1.042342,2.056646,0.275858,0.692118,1.851789,2.801575,0.063158,0.370801,15
13,2,wide,"[256, 128, 64]",0.0010,0.15,0.75,1.047239,2.061054,0.272750,0.691997,1.865580,2.804763,0.061024,0.366659,20


## Training on the best setup

Train longer with the winner from the search above.

In [ ]:
best = search_df.iloc[0]
best_layers = eval(best["layers"])

print("Best setup:", best["name"], best["layers"], "lr=", best["lr"], "dropout=", best["dropout"])

torch.manual_seed(RANDOM_SEED)
best_model = DecomposedFPLNetConfig(in_dim, best_layers, dropout=float(best["dropout"]))

best_model, val_pred_tuned, tuned_metrics, _ = train_decomposed_config(
    best_model,
    X_train_s,
    y_train,
    X_val_s,
    y_val,
    val_mask,
    epochs=100,
    batch_size=4096,
    lr=float(best["lr"]),
    patience=12,
    aux_points_weight=float(best["aux_weight"]),
    verbose=True,
)

tuned_metrics["model"] = "mlp_decomposed_tuned"

print("\nBefore tuning vs after heavy training:")
cols = [c for c in ["model", "spearman_all", "mae_all", "spearman_played", "mae_played", "epochs_ran"] if c in pd.DataFrame([decomp_metrics, tuned_metrics]).columns]
pd.DataFrame([decomp_metrics, tuned_metrics])[cols]

_final_specs = {
    "roll5_points": "total_points_roll5",
    "last_season_ppg": "last_season_ppg",
    "mlp_decomposed_tuned": val_pred_tuned,
}
if BASELINE_VAL_PREDS_PATH.exists():
    _base = pd.read_csv(BASELINE_VAL_PREDS_PATH)
    _final_specs["mlp_points"] = _base["pred_mlp"].to_numpy()
    _final_specs["ridge_linear"] = _base["pred_ridge"].to_numpy()
comparison_final = build_comparison_table(val_df, y_val["total_points"], val_mask, _final_specs)

comparison_final.to_csv(RESULTS_DIR / "phase3_model_comparison_with_decomposed.csv", index=False)
print("\nSaved fair comparison (tuned decomposed vs baselines):")
print(comparison_final[["model", "mae_all", "spearman_all", "mae_played", "spearman_played"]].to_string(index=False))

search_df.to_csv(RESULTS_DIR / "phase3_decomposed_search.csv", index=False)

val_preds_tuned = val_df[["season", "element", "gw", "player_id", "name", "minutes", TARGET]].copy()
val_preds_tuned["pred_decomposed_tuned"] = val_pred_tuned
val_preds_tuned.to_csv(RESULTS_DIR / "val_2024_25_predictions_decomposed_tuned.csv", index=False)

torch.save(
    {
        "model_state": best_model.state_dict(),
        "name": best["name"],
        "layers": best_layers,
        "lr": float(best["lr"]),
        "dropout": float(best["dropout"]),
        "aux_weight": float(best["aux_weight"]),
        "in_dim": in_dim,
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
    },
    RESULTS_DIR / "phase3_decomposed_tuned.pt",
)
print("Saved to", RESULTS_DIR)

Best setup: wide [256, 128, 64] lr= 0.001 dropout= 0.25
  rho_all=0.6941 MAE_all=1.0388  epochs=41

Before tuning vs after heavy training:

Saved fair comparison (tuned decomposed vs baselines):
               model  mae_all  spearman_all  mae_played  spearman_played
        roll5_points 1.090377      0.675553    2.081290         0.290270
     last_season_ppg 1.552684      0.326069    2.188108         0.226227
mlp_decomposed_tuned 1.038826      0.694113    1.868526         0.368942
          mlp_points 1.056206      0.686446    1.863091         0.366992
        ridge_linear 1.114626      0.644458    1.870431         0.347047
Saved to C:\FPL_project\results
